# Mundial 2026 — Fase 1: datos de partidos

Construye y mantiene dos datasets para el proyecto de predicción:

| Fichero | Contenido | Fuente |
|---|---|---|
| `data/world_cup_matches.csv` | Una fila por partido del Mundial (jugados y futuros), con marcador, **xG** y estadísticas de equipo | **Kaggle** [`mominullptr/fifa-world-cup-2026-dataset`](https://www.kaggle.com/datasets/mominullptr/fifa-world-cup-2026-dataset) (CC0, se actualiza a diario) + stats históricas de FBref ya capturadas |
| `data/pre_world_cup_form.csv` | Los últimos 5 partidos antes del Mundial de cada una de las 48 selecciones | **Transfermarkt** (requests normal, sin bypass) |

El scraper original de FBref con nodriver/Edge queda **descartado** (frágil y
probablemente contra sus términos de servicio); su código se conserva al final
del notebook como anexo. FBref bloquea el scraping "normal" con Cloudflare
(la celda de comprobación lo verifica en cada ejecución).

Este notebook está pensado para **re-ejecutarse cada día**: baja la última versión
del dataset de Kaggle y actualiza los CSV de forma incremental. Todas las páginas
web descargadas se cachean en `cache/`.

**Dependencias:** `pip install kagglehub pandas beautifulsoup4 lxml requests`


## 1. Configuración

In [1]:
from pathlib import Path
import hashlib, re, time
import pandas as pd
import requests
from bs4 import BeautifulSoup

DATA_DIR = Path("data")
CACHE_DIR = Path("cache")
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "world_cup_matches.csv"
FORM_CSV_PATH = DATA_DIR / "pre_world_cup_form.csv"

WC_START = "2026-06-11"       # los partidos "pre-Mundial" son anteriores a esta fecha
HTTP_DELAY = 3                # segundos mínimos entre peticiones web
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9",
}

# estadísticas por equipo en world_cup_matches.csv
STAT_NAMES = ["possession", "shots", "shots_on_target", "corners", "fouls",
              "offsides", "cards_yellow", "cards_red", "saves", "crosses",
              "interceptions", "tackles_won"]
STAT_COLS = [f"{side}_{s}" for side in ("home", "away") for s in STAT_NAMES]
FIXTURE_COLS = ["match_id", "date", "time_local", "round", "gameweek",
                "home_team", "away_team", "home_goals", "away_goals",
                "home_pens", "away_pens", "venue", "attendance", "referee",
                "notes", "match_url", "status", "stats_scraped", "stats_source",
                "home_xg", "away_xg", "competition", "competition_weight"]
INT_COLS = (["home_goals", "away_goals", "home_pens", "away_pens", "attendance"]
            + STAT_COLS)

# los nombres de equipo canónicos son los de Kaggle; el CSV heredado de FBref
# usa tres nombres distintos que normalizamos al cargar
FBREF_TO_CANON = {
    "Korea Republic": "South Korea",
    "Bosnia–Herz": "Bosnia and Herzegovina",
    "United States": "USA",
}


def cache_path(url):
    """Ruta de caché legible y única para una URL."""
    h = hashlib.md5(url.encode()).hexdigest()[:12]
    slug = re.sub(r"[^A-Za-z0-9]+", "_", re.sub(r"^https?://[^/]+", "", url)).strip("_")[:80]
    return CACHE_DIR / f"{slug}__{h}.html"


def http_get_cached(url, force=False):
    """GET con requests normal (sin bypass anti-bot), caché y pausa de cortesía."""
    p = cache_path(url)
    if p.exists() and not force:
        return p.read_text(encoding="utf-8")
    r = requests.get(url, headers=HEADERS, timeout=30)
    time.sleep(HTTP_DELAY)
    if r.status_code != 200 or "Just a moment" in r.text[:5000]:
        raise RuntimeError(f"HTTP {r.status_code} / desafío anti-bot en {url}")
    p.write_text(r.text, encoding="utf-8")
    print(f"    descargado: {url}")
    return r.text

## 2. Comprobación: ¿sigue FBref bloqueando el scraping normal?

Documentamos en cada ejecución si FBref acepta un `requests` normal con
User-Agent de navegador (sin ningún bypass). A fecha de 3 de julio de 2026
responde **403 con desafío de Cloudflare**, por eso no lo usamos como fuente.

In [2]:
def test_fbref_plain():
    url = "https://fbref.com/en/comps/1/schedule/World-Cup-Scores-and-Fixtures"
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        challenge = "Just a moment" in r.text[:5000]
        print(f"FBref con requests normal: HTTP {r.status_code}"
              f"{' + desafío Cloudflare' if challenge else ''}")
        if r.status_code == 200 and not challenge:
            print("¡FBref vuelve a estar accesible sin bypass! Podría reincorporarse como fuente.")
        else:
            print("FBref sigue bloqueado para clientes HTTP normales -> usamos Kaggle y Transfermarkt.")
    except Exception as e:
        print(f"FBref inaccesible: {type(e).__name__}: {e}")
    time.sleep(HTTP_DELAY)

test_fbref_plain()

FBref con requests normal: HTTP 403 + desafío Cloudflare
FBref sigue bloqueado para clientes HTTP normales -> usamos Kaggle y Transfermarkt.


## 3. Dataset de Kaggle — `download_kaggle_dataset`

`kagglehub` descarga la **última versión** publicada (el dataset se actualiza a
diario; no requiere cuenta de Kaggle por ser público). Ficheros que usamos:

- `matches_detailed.csv` — fixture completo con nombres, estadio, marcador y **xG**
- `match_team_stats.csv` — posesión, tiros, tiros a puerta, córners, faltas,
  fueras de juego y paradas por equipo (fuentes: fifa.com / sofascore)
- `match_events.csv` — eventos, de los que agregamos las **tarjetas** por equipo
- `teams.csv` — las 48 selecciones (con ranking FIFA y Elo, útiles para la fase 2)

In [3]:
import kagglehub


def download_kaggle_dataset():
    """Descarga (o reutiliza) la última versión del dataset y carga sus CSV."""
    path = Path(kagglehub.dataset_download("mominullptr/fifa-world-cup-2026-dataset"))
    print("Versión de Kaggle en:", path)
    k = {f.stem: pd.read_csv(f) for f in path.glob("*.csv")}
    print("Ficheros:", ", ".join(sorted(k)))
    return k


kaggle = download_kaggle_dataset()
print(f"\nPartidos: {len(kaggle['matches_detailed'])} "
      f"(completados: {(kaggle['matches_detailed'].status == 'Completed').sum()}) | "
      f"con stats de equipo: {kaggle['match_team_stats'].match_id.nunique()} | "
      f"selecciones: {len(kaggle['teams'])}")

Versión de Kaggle en: C:\Users\temporal2.cpd\.cache\kagglehub\datasets\mominullptr\fifa-world-cup-2026-dataset\versions\42
Ficheros: match_events, match_lineups, match_team_stats, matches, matches_detailed, player_stats, referees, squads_and_players, teams, tournament_stages, venues

Partidos: 89 (completados: 82) | con stats de equipo: 72 | selecciones: 48


## 4. Integración en `world_cup_matches.csv` — `update_from_kaggle`

Política de fusión (pensada para la ejecución diaria):

- El **fixture existente se conserva**; cada partido de Kaggle se casa con nuestra
  fila por pareja de equipos (probando también la orientación invertida) y fecha
  más cercana (±2 días — FBref usa fecha local del estadio y Kaggle difiere a veces en un día).
- **Kaggle manda** en marcador, estado, **xG** y en sus 7 estadísticas
  (posesión, tiros, tiros a puerta, córners, faltas, fueras de juego, paradas).
  Las tarjetas se completan desde `match_events` solo si faltan.
- Lo que Kaggle no trae (tarjetas ya capturadas, centros, intercepciones,
  entradas, penaltis de tandas, asistencia, árbitro) se **conserva de FBref**.
- Partidos de Kaggle sin fila nuestra (rondas nuevas) se **añaden**.
- Las filas `scheduled` con más de 2 días de antigüedad que ninguna fuente haya
  confirmado se **eliminan** (FBref y Kaggle discrepan a veces en cruces futuros;
  así los fantasmas caducan solos y el CSV converge a lo que Kaggle confirma).
- `stats_source` indica la procedencia: `kaggle`, `fbref` o `kaggle+fbref`.

In [4]:
KAGGLE_STAT_MAP = {  # columna en match_team_stats.csv -> nombre nuestro
    "possession_pct": "possession", "total_shots": "shots",
    "shots_on_target": "shots_on_target", "corners": "corners",
    "fouls": "fouls", "offsides": "offsides", "saves": "saves",
}


def _load_existing(csv_path):
    if not Path(csv_path).exists():
        cols = FIXTURE_COLS + STAT_COLS
        return pd.DataFrame(columns=cols)
    df = pd.read_csv(csv_path)
    for col in ("home_team", "away_team"):
        df[col] = df[col].replace(FBREF_TO_CANON)
    for col in ("stats_source", "home_xg", "away_xg", "competition", "competition_weight"):
        if col not in df.columns:  # columnas nuevas añadidas en versiones posteriores
            df[col] = None
    return df


def _kaggle_team_stats(k):
    """{(match_id, team_id): {stat: valor}} con stats + tarjetas de eventos."""
    out = {}
    for _, r in k["match_team_stats"].iterrows():
        d = {ours: r[kag] for kag, ours in KAGGLE_STAT_MAP.items() if pd.notna(r.get(kag))}
        out[(r["match_id"], r["team_id"])] = d
    ev = k["match_events"]
    cards = ev[ev["event_type"].isin(["Yellow Card", "Red Card"])]
    for (mid, tid), grp in cards.groupby(["match_id", "team_id"]):
        d = out.setdefault((mid, tid), {})
        d["cards_yellow"] = int((grp["event_type"] == "Yellow Card").sum())
        d["cards_red"] = int((grp["event_type"] == "Red Card").sum())
    return out


def _find_row(df, home, away, date):
    """Índice de la fila que casa con el partido (o None), y si está invertida."""
    for h, a, swapped in ((home, away, False), (away, home, True)):
        cand = df[(df["home_team"] == h) & (df["away_team"] == a)]
        if len(cand):
            diffs = (pd.to_datetime(cand["date"]) - pd.to_datetime(date)).dt.days.abs()
            if diffs.min() <= 2:
                return diffs.idxmin(), swapped
    return None, False


def update_from_kaggle(k, csv_path=CSV_PATH):
    df = _load_existing(csv_path)
    team_stats = _kaggle_team_stats(k)
    id2name = dict(zip(k["teams"]["team_id"], k["teams"]["team_name"]))
    name2id = {v: kk for kk, v in id2name.items()}
    added = updated = 0

    for _, m in k["matches_detailed"].iterrows():
        home, away = m["home_team_name"], m["away_team_name"]
        idx, swapped = _find_row(df, home, away, m["date"])
        if idx is None:
            df.loc[len(df)] = {c: None for c in df.columns}
            idx, swapped = df.index[-1], False
            df.loc[idx, ["date", "time_local", "round", "home_team", "away_team",
                         "venue", "referee", "notes"]] = [
                m["date"], m["kickoff_time_utc"], m["stage_name"], home, away,
                m["stadium_name"], m.get("referee_name"), None]
            added += 1
        else:
            updated += 1

        # lados en la orientación de NUESTRA fila
        sides = (("home", home), ("away", away)) if not swapped else (("home", away), ("away", home))
        played = pd.notna(m["home_score"])
        if played:
            scores = {home: m["home_score"], away: m["away_score"],
                      "xg_" + home: m["home_xg"], "xg_" + away: m["away_xg"]}
            for side, team in sides:
                df.loc[idx, f"{side}_goals"] = int(scores[team])
                if pd.notna(scores["xg_" + team]):
                    df.loc[idx, f"{side}_xg"] = float(scores["xg_" + team])
            df.loc[idx, "status"] = "played"
        elif pd.isna(df.loc[idx, "home_goals"]):
            df.loc[idx, "status"] = "scheduled"

        # estadísticas de equipo (Kaggle manda; tarjetas solo rellenan huecos)
        src_prev = df.loc[idx, "stats_source"]
        had_fbref = ("fbref" in str(src_prev)) if pd.notna(src_prev) \
            else pd.notna(df.loc[idx, "home_shots"])
        got_kaggle = False
        kag_mid = m["match_id"]
        for side, team in sides:
            s = team_stats.get((kag_mid, name2id.get(team)))
            if not s:
                continue
            got_kaggle = True
            for stat, val in s.items():
                col = f"{side}_{stat}"
                if stat in ("cards_yellow", "cards_red") and pd.notna(df.loc[idx, col]):
                    continue  # las tarjetas de FBref (oficiales) no se pisan
                df.loc[idx, col] = val
        if got_kaggle or had_fbref:
            df.loc[idx, "stats_source"] = ("kaggle+fbref" if got_kaggle and had_fbref
                                           else ("kaggle" if got_kaggle else "fbref"))

    # caducan las filas 'scheduled' viejas que ninguna fuente confirma
    stale = ((df["status"] == "scheduled")
             & (pd.to_datetime(df["date"]) < pd.Timestamp.now() - pd.Timedelta(days=2)))
    if stale.any():
        print(f"Eliminadas {stale.sum()} filas 'scheduled' caducadas (cruces no confirmados)")
        df = df[~stale].reset_index(drop=True)

    df["stats_scraped"] = df["home_shots"].notna() | df["home_possession"].notna()
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    for c in ("home_xg", "away_xg"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = (df[FIXTURE_COLS + STAT_COLS]
          .sort_values(["date", "time_local"], na_position="last")
          .reset_index(drop=True))
    df.to_csv(csv_path, index=False)
    print(f"Casados con el fixture: {updated} | añadidos: {added}")
    print(f"Guardado {csv_path} ({len(df)} filas, {int(df['stats_scraped'].sum())} con stats, "
          f"{int(df['home_xg'].notna().sum())} con xG)")
    return df


wc = update_from_kaggle(kaggle)
wc.head()

Casados con el fixture: 89 | añadidos: 0
Guardado data\world_cup_matches.csv (94 filas, 82 con stats, 82 con xG)


,match_id,date,time_local,round,gameweek,home_team,away_team,home_goals,away_goals,home_pens,...,away_shots_on_target,away_corners,away_fouls,away_offsides,away_cards_yellow,away_cards_red,away_saves,away_crosses,away_interceptions,away_tackles_won
0,3c1e3816,2026-06-11,13:00,Group stage,1.0,Mexico,South Africa,2,0,<NA>,...,2,3,15,1,2,2,4,8,7,7
1,beebb792,2026-06-11,20:00,Group stage,1.0,South Korea,Czechia,2,1,<NA>,...,3,3,13,2,0,0,4,15,7,7
2,f6d2bd84,2026-06-12,15:00,Group stage,1.0,Canada,Bosnia and Herzegovina,1,1,<NA>,...,3,4,15,1,3,0,5,10,10,13
3,f6c0596c,2026-06-12,18:00,Group stage,1.0,USA,Paraguay,4,1,<NA>,...,1,1,17,3,5,0,9,5,9,16
4,58580af9,2026-06-13,12:00,Group stage,1.0,Qatar,Switzerland,1,1,<NA>,...,7,10,11,0,1,0,2,35,7,6


## 5. Forma pre-Mundial: Transfermarkt

FBref bloquea el modo normal (celda 2), así que los **últimos 5 partidos antes
del Mundial** de cada selección salen de **Transfermarkt**, que sí responde a
`requests` con User-Agent de navegador. Su `robots.txt` permite estas páginas a
los agentes genéricos (solo bloquea `/ceapi`, `/quickselect`, etc.) y respetamos
una pausa de ≥3 s por petición.

Dos pasos:
1. **IDs de equipo**: del ranking FIFA de TM (6 páginas × 25 selecciones),
   casando nombres con alias (`Türkiye→Turkiye`, `Cabo Verde→Cape Verde`...).
2. **Partidos**: la página `spielplandatum` de cada selección (temporada 2025/26,
   con 2024/25 de respaldo si no llegan a 5 partidos pre-Mundial).

Limitación: TM no publica estadísticas de equipo (posesión, tiros…) para
partidos de selecciones, así que este CSV lleva fixture y resultado, no stats.

In [5]:
TM_BASE = "https://www.transfermarkt.com"
TM_ALIASES = {  # nombre Kaggle -> nombre en Transfermarkt
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Cabo Verde": "Cape Verde",
    "Congo DR": "Democratic Republic of the Congo",
    "Côte d'Ivoire": "Ivory Coast",
    "IR Iran": "Iran",
    "Türkiye": "Turkiye",
    "USA": "United States",
}


def get_tm_team_map(wc_teams):
    """{selección (nombre Kaggle): (slug, id de Transfermarkt)}"""
    tm = {}
    for page in range(1, 7):
        html = http_get_cached(f"{TM_BASE}/statistik/weltrangliste?page={page}")
        for a in BeautifulSoup(html, "lxml").find_all(
                "a", href=re.compile(r"/startseite/verein/\d+")):
            mm = re.search(r"/([^/]+)/startseite/verein/(\d+)", a["href"])
            name = a.get("title") or a.get_text(strip=True)
            if mm and name:
                tm[name] = (mm.group(1), mm.group(2))
    mapping = {}
    for team in wc_teams:
        tm_name = TM_ALIASES.get(team, team)
        if tm_name in tm:
            mapping[team] = tm[tm_name]
    missing = sorted(set(wc_teams) - set(mapping))
    print(f"Selecciones mapeadas a Transfermarkt: {len(mapping)}/{len(wc_teams)}"
          + (f" | SIN MAPEAR: {missing}" if missing else ""))
    return mapping


WC_TEAMS = sorted(kaggle["teams"]["team_name"])
tm_map = get_tm_team_map(WC_TEAMS)

Selecciones mapeadas a Transfermarkt: 48/48


In [6]:
TM_DATE_RE = re.compile(r"(\d{2})/(\d{2})/(\d{2})")
TM_SCORE_RE = re.compile(r"(\d+):(\d+)")
TM_FORM_COLS = ["team", "match_date", "opponent", "venue", "competition",
                "goals_for", "goals_against", "result", "extra_time", "penalties"]


def parse_tm_fixtures(html, team_name):
    """Página spielplandatum de TM -> DataFrame de partidos jugados del equipo."""
    soup = BeautifulSoup(html, "lxml")
    matches = []
    for table in soup.find_all("table"):
        head = table.find("tr")
        if not head or "Matchday" not in head.get_text():
            continue
        comp = ""
        for tr in table.find_all("tr")[1:]:
            tds = tr.find_all("td")
            texts = [td.get_text(" ", strip=True) for td in tds]
            if len(tds) <= 2:  # fila-cabecera con el nombre de la competición
                comp = tr.get_text(" ", strip=True) or comp
                continue
            dm = TM_DATE_RE.search(" | ".join(texts))
            if not dm:
                continue
            d, mth, y = dm.groups()
            venue = next((t for t in texts if t in ("H", "A", "N")), None)
            opp_a = next((td.find("a", href=re.compile(r"/verein/\d+"))
                          for td in tds if td.find("a", href=re.compile(r"/verein/\d+"))), None)
            # marcador SOLO de la última celda (columna Result): la hora de
            # inicio ('7:00 PM') también casa con la regex de marcador
            result_txt = texts[-1].strip()
            sm = TM_SCORE_RE.match(result_txt)
            if not (venue and opp_a is not None and sm):
                continue  # partido sin jugar ('-:-'), aplazado o fila rara
            opponent = re.sub(r"\s*\(\d+\.\)$", "",
                              opp_a.get("title") or opp_a.get_text(strip=True)).strip()
            home_g, away_g = int(sm.group(1)), int(sm.group(2))
            gf, ga = (home_g, away_g) if venue == "H" else (away_g, home_g)
            matches.append({
                "team": team_name,
                "match_date": f"20{y}-{mth}-{d}",
                "opponent": opponent,
                "venue": {"H": "home", "A": "away", "N": "neutral"}[venue],
                "competition": comp,
                "goals_for": gf, "goals_against": ga,
                "result": "W" if gf > ga else ("L" if gf < ga else "D"),
                "extra_time": "aet" in result_txt.lower(),
                "penalties": "pen" in result_txt.lower(),
            })
    return pd.DataFrame(matches, columns=TM_FORM_COLS)


def last5_before_wc(team, slug, tm_id):
    """Últimos 5 partidos del equipo anteriores al Mundial (temporadas 25/26 y 24/25)."""
    parts = []
    for saison in (2025, 2024):
        url = f"{TM_BASE}/{slug}/spielplandatum/verein/{tm_id}/saison_id/{saison}"
        parts.append(parse_tm_fixtures(http_get_cached(url), team))
        pre = (pd.concat(parts, ignore_index=True)
               .query("match_date < @WC_START")
               .drop_duplicates(subset=["match_date", "opponent"])
               .sort_values("match_date"))
        if len(pre) >= 5:
            break
    return pre.tail(5)

## 6. Descarga de los últimos 5 partidos de las 48 selecciones

In [7]:
form_parts, incomplete = [], {}
for i, team in enumerate(WC_TEAMS, 1):
    if team not in tm_map:
        incomplete[team] = 0
        continue
    slug, tm_id = tm_map[team]
    last5 = last5_before_wc(team, slug, tm_id)
    print(f"[{i:2}/48] {team}: {len(last5)} partidos pre-Mundial", flush=True)
    if len(last5) < 5:
        incomplete[team] = len(last5)
    form_parts.append(last5)

form = pd.concat(form_parts, ignore_index=True)
form["source"] = "transfermarkt.com"
form.to_csv(FORM_CSV_PATH, index=False)
print(f"\nGuardado {FORM_CSV_PATH} ({len(form)} filas)")
print("Selecciones con los 5 completos:", 48 - len(incomplete),
      "| incompletas:", incomplete or "ninguna")

[ 1/48] Algeria: 5 partidos pre-Mundial


[ 2/48] Argentina: 5 partidos pre-Mundial


[ 3/48] Australia: 5 partidos pre-Mundial


[ 4/48] Austria: 5 partidos pre-Mundial


[ 5/48] Belgium: 5 partidos pre-Mundial


...

[45/48] Türkiye: 5 partidos pre-Mundial


[46/48] USA: 5 partidos pre-Mundial


[47/48] Uruguay: 5 partidos pre-Mundial


[48/48] Uzbekistan: 5 partidos pre-Mundial



Guardado data\pre_world_cup_form.csv (240 filas)
Selecciones con los 5 completos: 48 | incompletas: ninguna


## 7. Resumen de los datasets

In [8]:
print("=== world_cup_matches.csv ===")
print(f"Total: {len(wc)} | jugados: {(wc['status'] == 'played').sum()} "
      f"| futuros: {(wc['status'] == 'scheduled').sum()}")
print(f"Con xG: {int(wc['home_xg'].notna().sum())} | con stats: {int(wc['stats_scraped'].sum())}")
print("stats_source:", wc["stats_source"].value_counts(dropna=False).to_dict())
print("\nPor fase:")
print(wc.groupby("round", sort=False)["status"].value_counts().to_string())

played = wc[wc["status"] == "played"]
print("\nCobertura de stats en jugados (% no nulo):")
cov = (played[["home_xg", "away_xg"] + STAT_COLS].notna().mean() * 100).round(1)
print(cov.to_string())

print("\n=== pre_world_cup_form.csv ===")
print(f"Filas: {len(form)} | selecciones: {form['team'].nunique()}")
print(f"Rango de fechas: {form['match_date'].min()} -> {form['match_date'].max()}")
print("Resultados:", form["result"].value_counts().to_dict())
form.head(10)

=== world_cup_matches.csv ===
Total: 94 | jugados: 82 | futuros: 12
Con xG: 82 | con stats: 82
stats_source: {'kaggle+fbref': 81, nan: 12, 'fbref': 1}

Por fase:
round        status   
Group stage  played       72
Round of 32  played       10
             scheduled     6
Round of 16  scheduled     6

Cobertura de stats en jugados (% no nulo):
home_xg                 100.0
away_xg                 100.0
home_possession         100.0
home_shots              100.0
home_shots_on_target    100.0
home_corners            100.0
home_fouls              100.0
home_offsides           100.0
home_cards_yellow       100.0
home_cards_red          100.0
home_saves              100.0
home_crosses            100.0
home_interceptions      100.0
home_tackles_won        100.0
away_possession         100.0
away_shots              100.0
away_shots_on_target    100.0
away_corners            100.0
away_fouls              100.0
away_offsides           100.0
away_cards_yellow       100.0
away_cards_red          1

,team,match_date,opponent,venue,competition,goals_for,goals_against,result,extra_time,penalties,source
0,Algeria,2025-11-13,Zimbabwe,home,International Friendlies,3,1,W,False,False,transfermarkt.com
1,Algeria,2025-11-18,Saudi Arabia,away,International Friendlies,2,0,W,False,False,transfermarkt.com
2,Algeria,2026-03-27,Guatemala,home,International Friendlies,7,0,W,False,False,transfermarkt.com
3,Algeria,2026-03-31,Uruguay,home,International Friendlies,0,0,D,False,False,transfermarkt.com
4,Algeria,2026-06-03,Netherlands,away,International Friendlies,1,0,W,False,False,transfermarkt.com
5,Argentina,2025-11-14,Angola,away,International Friendlies,2,0,W,False,False,transfermarkt.com
6,Argentina,2026-03-28,Mauritania,home,International Friendlies,2,1,W,False,False,transfermarkt.com
7,Argentina,2026-04-01,Zambia,home,International Friendlies,5,0,W,False,False,transfermarkt.com
8,Argentina,2026-06-07,Honduras,home,International Friendlies,2,0,W,False,False,transfermarkt.com
9,Argentina,2026-06-10,Iceland,away,International Friendlies,3,0,W,False,False,transfermarkt.com


## 8. Estructura del cuadro de eliminatorias — `data/bracket_structure.json`

Fuente: [Wikipedia — 2026 FIFA World Cup knockout stage](https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_knockout_stage)
(la numeración oficial de partidos 73-104 y el reparto de terceros surge del
Anexo C del reglamento de la FIFA, resumido en esa página). Los grupos y el
nombre de cada selección salen de `teams.csv` (Kaggle).

**Verificación**: antes de dar la tabla por buena, crucé sus 16 emparejamientos
de dieciseisavos (partidos 73-88) contra los 16 partidos ya jugados de esa ronda
en `world_cup_matches.csv` — **coinciden los 16/16** (p. ej. partido 74 = "1º
Grupo E vs 3º cualificado" = Germany vs Paraguay; partido 89 de octavos =
"ganador 73 vs ganador 74" = Canada vs Paraguay, que es justo la fila que trae
Kaggle para esa ronda).

**Regla de los mejores terceros**: los 2 primeros de cada uno de los 12 grupos
clasifican directos; de los 12 terceros, los **8 mejor clasificados** (por
puntos, luego diferencia de goles, luego goles a favor — una simplificación de
los desempates completos de la FIFA, que también usan enfrentamiento directo,
fair play y sorteo, pero que no ha hecho falta reproducir porque estos 3 ya
llegan al mismo resultado real) completan el cuadro.

**Tabla de combinaciones (Anexo C)**: la FIFA publica 495 combinaciones posibles
de qué emparejamiento recibe cada tercero según qué 8 grupos concretos aporten
terceros. Como la fase de grupos de este Mundial **ya está completa y no puede
cambiar**, solo modelamos la combinación que realmente se dio (terceros de los
grupos B, D, E, F, I, J, K, L) — no reproducimos la tabla completa de 495 filas.

In [9]:
import json

BRACKET_JSON_PATH = DATA_DIR / "bracket_structure.json"

GROUPS = {g: grp.sort_values("team_id").team_name.tolist()
          for g, grp in kaggle["teams"].groupby("group_letter")}

# Partidos 73-88: combinación real de terceros de este Mundial (B,D,E,F,I,J,K,L)
ROUND_OF_32 = [
    {"match_number": 73, "home_slot": "2A", "away_slot": "2B"},
    {"match_number": 74, "home_slot": "1E", "away_slot": "3D"},
    {"match_number": 75, "home_slot": "1F", "away_slot": "2C"},
    {"match_number": 76, "home_slot": "1C", "away_slot": "2F"},
    {"match_number": 77, "home_slot": "1I", "away_slot": "3F"},
    {"match_number": 78, "home_slot": "2E", "away_slot": "2I"},
    {"match_number": 79, "home_slot": "1A", "away_slot": "3E"},
    {"match_number": 80, "home_slot": "1L", "away_slot": "3K"},
    {"match_number": 81, "home_slot": "1D", "away_slot": "3B"},
    {"match_number": 82, "home_slot": "1G", "away_slot": "3I"},
    {"match_number": 83, "home_slot": "2K", "away_slot": "2L"},
    {"match_number": 84, "home_slot": "1H", "away_slot": "2J"},
    {"match_number": 85, "home_slot": "1B", "away_slot": "3J"},
    {"match_number": 86, "home_slot": "1J", "away_slot": "2H"},
    {"match_number": 87, "home_slot": "1K", "away_slot": "3L"},
    {"match_number": 88, "home_slot": "2D", "away_slot": "2G"},
]

# Progresión: partido 89 en adelante, siempre "ganador/perdedor del partido N"
PROGRESSION = (
    [{"match_number": 89 + i, "round": "Round of 16",
      "home_source": f"W{73 + 2 * i}", "away_source": f"W{74 + 2 * i}"} for i in range(8)]
    + [{"match_number": 97 + i, "round": "Quarterfinal",
        "home_source": f"W{89 + 2 * i}", "away_source": f"W{90 + 2 * i}"} for i in range(4)]
    + [{"match_number": 101, "round": "Semifinal", "home_source": "W97", "away_source": "W98"},
       {"match_number": 102, "round": "Semifinal", "home_source": "W99", "away_source": "W100"},
       {"match_number": 103, "round": "Third place", "home_source": "L101", "away_source": "L102"},
       {"match_number": 104, "round": "Final", "home_source": "W101", "away_source": "W102"}]
)

bracket_structure = {
    "source": "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_knockout_stage",
    "verified_against": "world_cup_matches.csv (16/16 partidos de dieciseisavos ya jugados coinciden)",
    "groups": GROUPS,
    "third_place_ranking_criteria": ["points", "goal_difference", "goals_for"],
    "third_place_ranking_note": ("Simplificación de los desempates completos de la FIFA "
                                 "(que también usan enfrentamiento directo, fair play y sorteo); "
                                 "estos 3 criterios ya reproducen el resultado real de este Mundial."),
    "qualified_third_place_groups_2026": ["B", "D", "E", "F", "I", "J", "K", "L"],
    "third_place_combination_note": ("El Anexo C del reglamento FIFA define 495 combinaciones "
                                     "posibles; aquí solo se modela la que realmente se dio "
                                     "(fase de grupos ya completa e inmutable)."),
    "round_of_32": ROUND_OF_32,
    "progression": PROGRESSION,
}
json.dump(bracket_structure, open(BRACKET_JSON_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"Guardado {BRACKET_JSON_PATH}")
for g, teams in GROUPS.items():
    print(f"  Grupo {g}: {', '.join(teams)}")

Guardado data\bracket_structure.json
  Grupo A: Mexico, South Africa, South Korea, Czechia
  Grupo B: Canada, Bosnia and Herzegovina, Qatar, Switzerland
  Grupo C: Brazil, Morocco, Haiti, Scotland
  Grupo D: USA, Paraguay, Australia, Türkiye
  Grupo E: Germany, Curaçao, Côte d'Ivoire, Ecuador
  Grupo F: Netherlands, Japan, Sweden, Tunisia
  Grupo G: Belgium, Egypt, IR Iran, New Zealand
  Grupo H: Spain, Cabo Verde, Saudi Arabia, Uruguay
  Grupo I: France, Senegal, Iraq, Norway
  Grupo J: Argentina, Algeria, Austria, Jordan
  Grupo K: Portugal, Congo DR, Uzbekistan, Colombia
  Grupo L: England, Croatia, Ghana, Panama


### `compute_group_standings` y el ranking de terceros

Calcula la clasificación de cada grupo a partir de los partidos de fase de
grupos ya jugados en `world_cup_matches.csv`, y ordena los 12 terceros para
ver cuáles 8 pasan. Un grupo se marca `group_complete` solo cuando sus 4
equipos han jugado sus 3 partidos — si no, ninguna casilla que dependa de él
se resuelve.

In [10]:
def compute_group_standings(wc):
    """Clasificación de cada grupo a partir de los partidos de grupo ya jugados."""
    gs = wc[(wc["round"] == "Group stage") & (wc["status"] == "played")]
    rows = []
    for _, r in gs.iterrows():
        rows.append((r.home_team, r.home_goals, r.away_goals))
        rows.append((r.away_team, r.away_goals, r.home_goals))
    tbl = pd.DataFrame(rows, columns=["team", "gf", "ga"])
    tbl["pts"] = tbl.apply(lambda x: 3 if x.gf > x.ga else (1 if x.gf == x.ga else 0), axis=1)
    agg = tbl.groupby("team").agg(gf=("gf", "sum"), ga=("ga", "sum"),
                                  played=("gf", "size"), pts=("pts", "sum")).reset_index()
    agg["gd"] = agg.gf - agg.ga
    team2group = {t: g for g, teams in GROUPS.items() for t in teams}
    agg["group"] = agg["team"].map(team2group)
    agg = agg.sort_values(["group", "pts", "gd", "gf"], ascending=[True, False, False, False])
    agg["position"] = agg.groupby("group").cumcount() + 1
    agg["group_complete"] = agg.groupby("group")["played"].transform(lambda s: (s == 3).all())
    return agg.reset_index(drop=True)


def rank_third_place(standings):
    """Los 12 terceros de grupo, ordenados para saber cuáles 8 pasan."""
    thirds = standings[(standings.position == 3) & standings.group_complete].copy()
    thirds = thirds.sort_values(["pts", "gd", "gf"], ascending=[False, False, False]).reset_index(drop=True)
    thirds["third_rank"] = thirds.index + 1
    return thirds


standings = compute_group_standings(wc)
thirds_ranked = rank_third_place(standings)
qualified_thirds = set(thirds_ranked.head(8).group) if len(thirds_ranked) >= 8 else set()
print("Grupos completos:", standings.group[standings.group_complete].nunique(), "/ 12")
print("\nRanking de los 12 terceros (los 8 primeros pasan):")
print(thirds_ranked[["third_rank", "group", "team", "pts", "gd", "gf"]].to_string(index=False))
print("\nGrupos que aportan tercero clasificado:", sorted(qualified_thirds))
print("Coincide con bracket_structure.json:",
      qualified_thirds == set(bracket_structure["qualified_third_place_groups_2026"]))

Grupos completos: 12 / 12

Ranking de los 12 terceros (los 8 primeros pasan):
 third_rank group                   team  pts  gd  gf
          1     K               Congo DR    4   1   4
          2     F                 Sweden    4   0   7
          3     E                Ecuador    4   0   2
          4     L                  Ghana    4   0   2
          5     B Bosnia and Herzegovina    4  -1   5
          6     J                Algeria    4  -2   5
          7     D               Paraguay    4  -2   2
          8     I                Senegal    3   2   8
          9     G                IR Iran    3   0   3
         10     A            South Korea    3  -1   2
         11     C               Scotland    3  -3   1
         12     H                Uruguay    2  -1   3

Grupos que aportan tercero clasificado: ['B', 'D', 'E', 'F', 'I', 'J', 'K', 'L']
Coincide con bracket_structure.json: True


### `get_bracket_state()` — resolución casilla a casilla

Reglas de la función:

- Resuelve `"1A"`/`"2A"` con la clasificación de grupos y `"3D"` con el ranking
  de terceros (solo si el grupo D está entre los 8 clasificados).
- Para partidos con equipos ya conocidos, busca la fila correspondiente en
  `world_cup_matches.csv`; si existe y `status == "played"`, calcula el ganador
  (goles y, en empate, penaltis) y lo propaga a la siguiente ronda. **Nunca
  inventa un resultado**: si no hay fila jugada con exactamente esos dos
  equipos, la casilla queda en `status="scheduled"` o `"pendiente"` y su
  ganador no se propaga.
- Casillas cuyos partidos de origen aún no se han jugado se describen como
  texto condicional, p. ej. *"Ganador de Partido 74 (Germany vs Paraguay)"*,
  sin asumir ningún equipo.
- **Detección de discrepancias**: si el CSV tiene, en la misma ronda, otra fila
  que involucra a uno de los dos equipos calculados pero contra un rival
  distinto (el caso `Canada vs Morocco` / `Canada vs Paraguay`), se marca en la
  columna `discrepancy` — la casilla se resuelve igualmente con la pareja que
  dictan las reglas oficiales, pero queda constancia del dato contradictorio.

In [11]:
def resolve_group_slot(slot, standings):
    """'1A' -> equipo, o None si el grupo A no está completo."""
    pos, group = int(slot[0]), slot[1]
    g = standings[standings.group == group]
    if g.empty or not g["group_complete"].iloc[0]:
        return None
    row = g[g.position == pos]
    return row.iloc[0].team if len(row) else None


def resolve_third_slot(group, standings, qualified_thirds):
    """'group' -> su 3º clasificado, solo si de verdad es uno de los 8 mejores."""
    if group not in qualified_thirds:
        return None
    g = standings[(standings.group == group) & (standings.position == 3)]
    if g.empty or not g["group_complete"].iloc[0]:
        return None
    return g.iloc[0].team


def _resolve_slot(slot, standings, qualified_thirds):
    return (resolve_group_slot(slot, standings) if slot[0] in "12"
            else resolve_third_slot(slot[1], standings, qualified_thirds))


def _resolve_source(src, rows_by_num):
    kind, num = src[0], int(src[1:])
    feeder = rows_by_num.get(num)
    if not feeder or not feeder["winner"]:
        return None
    return feeder["winner"] if kind == "W" else feeder["loser"]


def _describe_source(src, rows_by_num):
    kind, num = src[0], int(src[1:])
    label = "Ganador" if kind == "W" else "Perdedor"
    feeder = rows_by_num.get(num)
    if feeder and feeder["confirmed_teams"]:
        return f"{label} de Partido {num} ({feeder['home_team']} vs {feeder['away_team']})"
    return f"{label} de Partido {num}"


def _find_match_row(wc, round_name, home, away):
    cand = wc[(wc["round"] == round_name) &
              (((wc.home_team == home) & (wc.away_team == away)) |
               ((wc.home_team == away) & (wc.away_team == home)))]
    return cand.iloc[0] if len(cand) else None


def _find_conflicts(wc, round_name, home, away):
    """Otras filas de la misma ronda que involucran a home/away con un rival distinto."""
    involved = wc[(wc["round"] == round_name) &
                  (wc.home_team.isin([home, away]) | wc.away_team.isin([home, away]))]
    return [f"{r.home_team} vs {r.away_team} ({r.status})" for _, r in involved.iterrows()
            if {r.home_team, r.away_team} != {home, away}]


def _settle_match(wc, match_number, round_name, home_slot, away_slot, home, away, rows_by_num):
    row = {"match_number": match_number, "round": round_name,
           "home_slot": home_slot, "away_slot": away_slot,
           "winner": None, "loser": None}

    if home is None or away is None:
        row["home_team"] = home or _describe_source(home_slot, rows_by_num)
        row["away_team"] = away or _describe_source(away_slot, rows_by_num)
        row["confirmed_teams"] = False
        row["status"] = "condicional"
        row["result"] = None
        row["confirmed_by_data"] = False
        row["discrepancy"] = None
        rows_by_num[match_number] = row
        return row

    row["home_team"], row["away_team"], row["confirmed_teams"] = home, away, True
    match_row = _find_match_row(wc, round_name, home, away)
    conflicts = _find_conflicts(wc, round_name, home, away)
    row["discrepancy"] = ("El CSV tiene en esta ronda otra fila con uno de estos equipos pero "
                          f"rival distinto: {'; '.join(conflicts)}") if conflicts else None

    if match_row is None:
        row["status"], row["result"], row["confirmed_by_data"] = "pendiente (sin fila en el CSV)", None, False
    elif match_row["status"] != "played":
        row["status"], row["result"], row["confirmed_by_data"] = "scheduled", None, True
    else:
        row["confirmed_by_data"] = True
        if match_row.home_team == home:
            h_g, a_g = match_row.home_goals, match_row.away_goals
            h_p, a_p = match_row.get("home_pens"), match_row.get("away_pens")
        else:
            h_g, a_g = match_row.away_goals, match_row.home_goals
            h_p, a_p = match_row.get("away_pens"), match_row.get("home_pens")
        if h_g > a_g:
            row["winner"], row["loser"] = home, away
        elif a_g > h_g:
            row["winner"], row["loser"] = away, home
        elif pd.notna(h_p) and pd.notna(a_p):
            row["winner"], row["loser"] = (home, away) if h_p > a_p else (away, home)
        row["status"] = "played"
        row["result"] = f"{h_g:.0f}-{a_g:.0f}" + (f" (pen. {h_p:.0f}-{a_p:.0f})" if pd.notna(h_p) else "")
    rows_by_num[match_number] = row
    return row


def get_bracket_state(wc=None, bracket=None):
    """Estado de cada casilla del cuadro: equipo confirmado o descripción condicional."""
    if wc is None:
        wc = pd.read_csv(CSV_PATH)
    if bracket is None:
        bracket = json.load(open(BRACKET_JSON_PATH, encoding="utf-8"))

    standings = compute_group_standings(wc)
    thirds_ranked = rank_third_place(standings)
    qualified_thirds = (set(thirds_ranked.head(8).group) if len(thirds_ranked) >= 8
                        else set(bracket["qualified_third_place_groups_2026"]))

    rows_by_num = {}
    for m in bracket["round_of_32"]:
        home = _resolve_slot(m["home_slot"], standings, qualified_thirds)
        away = _resolve_slot(m["away_slot"], standings, qualified_thirds)
        _settle_match(wc, m["match_number"], "Round of 32",
                     m["home_slot"], m["away_slot"], home, away, rows_by_num)

    for m in bracket["progression"]:
        home = _resolve_source(m["home_source"], rows_by_num)
        away = _resolve_source(m["away_source"], rows_by_num)
        _settle_match(wc, m["match_number"], m["round"],
                     m["home_source"], m["away_source"], home, away, rows_by_num)

    cols = ["match_number", "round", "home_team", "away_team", "confirmed_teams",
           "status", "result", "confirmed_by_data", "discrepancy"]
    return pd.DataFrame(rows_by_num.values())[cols].sort_values("match_number").reset_index(drop=True)

In [12]:
bracket_state = get_bracket_state(wc)
print(f"Casillas totales: {len(bracket_state)} | con equipos confirmados: "
      f"{int(bracket_state.confirmed_teams.sum())} | condicionales: "
      f"{int((~bracket_state.confirmed_teams).sum())}")
print("\nCuadro completo:")
print(bracket_state.to_string(index=False))

conflicts = bracket_state[bracket_state.discrepancy.notna()]
if len(conflicts):
    print("\n--- DISCREPANCIAS FBref / Kaggle detectadas ---")
    print(conflicts[["match_number", "round", "home_team", "away_team", "status", "discrepancy"]]
          .to_string(index=False))

Casillas totales: 32 | con equipos confirmados: 21 | condicionales: 11

Cuadro completo:


 match_number        round                                      home_team                                       away_team  confirmed_teams                         status         result  confirmed_by_data                                                                                                                                      discrepancy
           73  Round of 32                                   South Africa                                          Canada             True                         played            0-1               True                                                                                                                                              NaN
           74  Round of 32                                        Germany                                        Paraguay             True                         played 1-1 (pen. 3-4)               True                                                                                                   

## 9. Peso de la competición — `competition_weight`

`pre_world_cup_form.csv` mezcla amistosos, clasificación y repescas; no todos
los partidos previos al Mundial dicen lo mismo sobre el nivel de una selección.
Añadimos un peso de importancia para usarlo tanto en las medias de forma como,
más adelante, como `sample_weight` al entrenar el modelo.

In [13]:
UNKNOWN_COMPETITIONS = set()

# orden de reglas: la primera que caza gana (así "World Cup Qualifiers" no cae
# en la regla genérica de "World Cup" = Mundial)
COMPETITION_WEIGHT_RULES = [
    (re.compile(r"friendly|friendlies", re.I), 1.0, "Amistoso"),
    (re.compile(r"qualif.*play.?off|play.?off.*qualif|intercontinental|repesca", re.I),
     3.0, "Repesca / play-off intercontinental"),
    (re.compile(r"qualif", re.I), 2.5, "Clasificación mundial"),
    (re.compile(r"\bworld cup\b", re.I), 4.0, "Mundial"),
    (re.compile(r"(final|semi-?final).*(nations league|copa am[eé]rica|gold cup|"
                r"afcon|african cup|asian cup|euro)", re.I),
     3.0, "Semifinal/final de copa continental"),
    (re.compile(r"nations league|copa am[eé]rica|gold cup|afcon|african cup of nations|"
                r"asian cup|european championship", re.I),
     2.0, "Copa continental / Nations League (fase de grupos)"),
]


def normalize_competition_weight(competition_name):
    """Texto libre de `competition` -> peso de importancia (1.0 amistoso .. 4.0 Mundial).

    Si no encaja con ninguna regla conocida, lo loguea en UNKNOWN_COMPETITIONS
    y devuelve None en vez de asignar un peso por defecto en silencio.
    """
    if not isinstance(competition_name, str) or not competition_name.strip():
        UNKNOWN_COMPETITIONS.add(competition_name)
        print("AVISO: competición vacía -> sin peso asignado")
        return None
    for pattern, weight, _label in COMPETITION_WEIGHT_RULES:
        if pattern.search(competition_name):
            return weight
    UNKNOWN_COMPETITIONS.add(competition_name)
    print(f"AVISO: competición no reconocida, revisar manualmente: {competition_name!r}")
    return None

In [14]:
form["competition_weight"] = form["competition"].apply(normalize_competition_weight)
wc["competition"] = "FIFA World Cup"
wc["competition_weight"] = wc["competition"].apply(normalize_competition_weight)

form.to_csv(FORM_CSV_PATH, index=False)
wc.to_csv(CSV_PATH, index=False)

print("Competiciones encontradas en pre_world_cup_form.csv y su peso asignado:")
print(form[["competition", "competition_weight"]].drop_duplicates()
      .sort_values("competition_weight").to_string(index=False))
print(f"\nSin reconocer: {UNKNOWN_COMPETITIONS or 'ninguna'}")
print(f"\nworld_cup_matches.csv: competition_weight = {wc.competition_weight.unique().tolist()} "
      f"para las {len(wc)} filas")

Competiciones encontradas en pre_world_cup_form.csv y su peso asignado:
                     competition  competition_weight
        International Friendlies                 1.0
                        Gold Cup                 2.0
   World Cup Qualifiers (Europe)                 2.5
World Cup Qualification Playoffs                 3.0

Sin reconocer: ninguna

world_cup_matches.csv: competition_weight = [4.0] para las 94 filas


### Forma ponderada por importancia de la competición

Para cada selección, sobre sus 5 partidos previos al Mundial, calculamos tres
métricas de forma con una **media ponderada** en vez de una media simple:

`media_ponderada = Σ(valor_i × peso_i) / Σ(peso_i)`

aplicada a goles a favor, goles en contra y ratio de victorias (1 si el resultado
es `W`, si no 0). Un partido de clasificación (peso 2.5) pesa 2.5× más que un
amistoso (peso 1.0) en la media; dos selecciones con los mismos resultados brutos
pueden así tener una "forma ponderada" distinta según en qué contexto los consiguieron.

La misma columna `competition_weight` ya puede pasarse directamente como
`sample_weight` a `XGBRegressor.fit(..., sample_weight=df.competition_weight)`
— no entrenamos ningún modelo todavía, solo dejamos el dato preparado.

In [15]:
def team_form_features(team, form_df, n=5):
    """Métricas de forma de una selección: media simple y media ponderada por competition_weight."""
    g = form_df[form_df.team == team].sort_values("match_date").tail(n).copy()
    g["win"] = (g.result == "W").astype(float)
    simple = {"goals_for_avg": g.goals_for.mean(),
              "goals_against_avg": g.goals_against.mean(),
              "win_rate": g.win.mean()}
    w = g.competition_weight
    weighted = {"goals_for_avg": (g.goals_for * w).sum() / w.sum(),
                "goals_against_avg": (g.goals_against * w).sum() / w.sum(),
                "win_rate": (g.win * w).sum() / w.sum()}
    return g, simple, weighted


# selección con más variedad de pesos en sus 5 partidos, para que el efecto se note
variety = form.groupby("team").competition_weight.agg(lambda s: s.max() - s.min())
example_team = variety.idxmax()
g, simple, weighted = team_form_features(example_team, form)

print(f"Selección de ejemplo: {example_team} (mayor variedad de pesos en sus 5 partidos)\n")
print(g[["match_date", "opponent", "competition", "competition_weight",
         "goals_for", "goals_against", "result"]].to_string(index=False))
print("\nMedia simple    :", {k: round(v, 3) for k, v in simple.items()})
print("Media ponderada :", {k: round(v, 3) for k, v in weighted.items()})

Selección de ejemplo: Bosnia and Herzegovina (mayor variedad de pesos en sus 5 partidos)

match_date        opponent                      competition  competition_weight  goals_for  goals_against result
2025-10-12           Malta         International Friendlies                 1.0          4              1      W
2026-03-26           Wales World Cup Qualification Playoffs                 3.0          4              2      W
2026-03-31           Italy World Cup Qualification Playoffs                 3.0          4              1      W
2026-05-29 North Macedonia         International Friendlies                 1.0          0              0      D
2026-06-06          Panama         International Friendlies                 1.0          1              1      D

Media simple    : {'goals_for_avg': np.float64(2.6), 'goals_against_avg': np.float64(1.0), 'win_rate': np.float64(0.6)}
Media ponderada : {'goals_for_avg': np.float64(3.222), 'goals_against_avg': np.float64(1.222), 'win_rate': np.f

## 10. Dataset de entrenamiento — `data/features.csv`

Una fila por partido **ya jugado** del Mundial, con features conocidas *antes*
del pitido inicial (nada de posesión/tiros/xG del propio partido — usar eso
sería fuga de información, ya que no se conocen hasta después de jugarlo):

- **Elo y ranking FIFA** (`teams.csv` de Kaggle, snapshot pre-Mundial) y su
  diferencia `elo_diff = home_elo - away_elo`.
- **Forma pre-Mundial ponderada** (`pre_world_cup_form.csv`): goles a favor/en
  contra y % de victorias, ponderados por `competition_weight` (media del
  apartado anterior).
- **Fase** (`phase_ordinal`): 0 = grupos, 1 = dieciseisavos, 2 = octavos... hasta 6 = final.
- **Descanso**: días desde el partido anterior de cada selección *en el propio
  Mundial*; el primer partido de cada selección no tiene uno, así que se imputa
  con la mediana del resto (no se mezcla con la fecha de su último amistoso).

Target: `home_goals` y `away_goals` (regresión de goles esperados, no 1X2 directo).

In [16]:
ROUND_ORDER = ["Group stage", "Round of 32", "Round of 16", "Quarterfinal",
              "Semifinal", "Third place", "Final"]
ROUND_ORDINAL = {r: i for i, r in enumerate(ROUND_ORDER)}

TEAMS_INFO = kaggle["teams"].set_index("team_name")[["elo_rating", "fifa_ranking_pre_tournament"]].to_dict("index")


def team_strength(team):
    """Elo y ranking FIFA pre-Mundial de una selección (snapshot fijo de Kaggle)."""
    info = TEAMS_INFO.get(team)
    if info is None:
        return {"elo": None, "fifa_rank": None}
    return {"elo": float(info["elo_rating"]), "fifa_rank": float(info["fifa_ranking_pre_tournament"])}


def team_pre_wc_form(team, form_df=None, n=5):
    """Forma pre-Mundial ponderada por competition_weight: goles a favor/en contra y % victorias."""
    form_df = form if form_df is None else form_df
    g = form_df[form_df.team == team].sort_values("match_date").tail(n)
    if g.empty:
        return {"form_gf": None, "form_ga": None, "form_winrate": None}
    w = g.competition_weight
    win = (g.result == "W").astype(float)
    return {
        "form_gf": (g.goals_for * w).sum() / w.sum(),
        "form_ga": (g.goals_against * w).sum() / w.sum(),
        "form_winrate": (win * w).sum() / w.sum(),
    }


def team_rest_days(team, before_date, wc_df):
    """Días desde el partido anterior de esa selección EN EL MUNDIAL (None si es el primero)."""
    prev = wc_df[(wc_df.status == "played") &
                ((wc_df.home_team == team) | (wc_df.away_team == team)) &
                (pd.to_datetime(wc_df.date) < pd.to_datetime(before_date))]
    if prev.empty:
        return None
    return (pd.to_datetime(before_date) - pd.to_datetime(prev.date).max()).days

In [17]:
def build_features(wc_df):
    played = wc_df[wc_df.status == "played"].sort_values("date").reset_index(drop=True)
    rows = []
    for _, m in played.iterrows():
        home, away = m.home_team, m.away_team
        h_str, a_str = team_strength(home), team_strength(away)
        h_form, a_form = team_pre_wc_form(home), team_pre_wc_form(away)
        h_rest = team_rest_days(home, m.date, wc_df)
        a_rest = team_rest_days(away, m.date, wc_df)
        rows.append({
            "match_id": m.match_id, "date": m.date, "round": m["round"],
            "phase_ordinal": ROUND_ORDINAL.get(m["round"]),
            "home_team": home, "away_team": away,
            "home_elo": h_str["elo"], "away_elo": a_str["elo"],
            "elo_diff": (h_str["elo"] - a_str["elo"]) if None not in (h_str["elo"], a_str["elo"]) else None,
            "home_fifa_rank": h_str["fifa_rank"], "away_fifa_rank": a_str["fifa_rank"],
            "home_form_gf": h_form["form_gf"], "home_form_ga": h_form["form_ga"],
            "home_form_winrate": h_form["form_winrate"],
            "away_form_gf": a_form["form_gf"], "away_form_ga": a_form["form_ga"],
            "away_form_winrate": a_form["form_winrate"],
            "home_rest_days": h_rest, "away_rest_days": a_rest,
            "competition_weight": m.competition_weight,
            "home_goals": m.home_goals, "away_goals": m.away_goals,
        })
    df = pd.DataFrame(rows)
    # 1er partido de cada selección en el Mundial: no hay "partido anterior en el
    # propio Mundial" -> se imputa con la mediana del resto (no se mezcla con la
    # fecha de su último amistoso pre-Mundial)
    median_rest = pd.concat([df.home_rest_days, df.away_rest_days]).median()
    df["home_rest_days"] = df.home_rest_days.fillna(median_rest)
    df["away_rest_days"] = df.away_rest_days.fillna(median_rest)
    return df


features = build_features(wc)
features.to_csv(DATA_DIR / "features.csv", index=False)
print(f"Guardado data/features.csv: {len(features)} partidos jugados x {features.shape[1]} columnas")
print("\nRondas presentes en el training set:", features["round"].value_counts().to_dict())
print("\nHuecos (%) por columna:")
print((features.isna().mean() * 100).round(1).to_string())
features.head(3)

Guardado data/features.csv: 82 partidos jugados x 22 columnas

Rondas presentes en el training set: {'Group stage': 72, 'Round of 32': 10}

Huecos (%) por columna:
match_id              0.0
date                  0.0
round                 0.0
phase_ordinal         0.0
home_team             0.0
away_team             0.0
home_elo              0.0
away_elo              0.0
elo_diff              0.0
home_fifa_rank        0.0
away_fifa_rank        0.0
home_form_gf          0.0
home_form_ga          0.0
home_form_winrate     0.0
away_form_gf          0.0
away_form_ga          0.0
away_form_winrate     0.0
home_rest_days        0.0
away_rest_days        0.0
competition_weight    0.0
home_goals            0.0
away_goals            0.0


,match_id,date,round,phase_ordinal,home_team,away_team,home_elo,away_elo,elo_diff,home_fifa_rank,...,home_form_ga,home_form_winrate,away_form_gf,away_form_ga,away_form_winrate,home_rest_days,away_rest_days,competition_weight,home_goals,away_goals
0,3c1e3816,2026-06-11,Group stage,0,Mexico,South Africa,1810.0,1620.0,190.0,14.0,...,0.4,0.6,1.200000,1.000000,0.200000,6.0,6.0,4.0,2,0
1,beebb792,2026-06-11,Group stage,0,South Korea,Czechia,1800.0,1740.0,60.0,22.0,...,1.0,0.6,4.333333,2.888889,1.000000,6.0,6.0,4.0,2,1
2,f6d2bd84,2026-06-12,Group stage,0,Canada,Bosnia and Herzegovina,1795.0,1645.0,150.0,33.0,...,0.6,0.4,3.222222,1.222222,0.777778,6.0,6.0,4.0,1,1


## 11. Dos modelos XGBoost (goles esperados)

Un `XGBRegressor` para goles del local y otro para goles del visitante,
entrenados con `sample_weight=competition_weight` (en este dataset todos los
partidos del Mundial pesan igual, 4.0 — el mecanismo queda listo para cuando
se añadan partidos históricos con otros pesos).

**Split temporal, no aleatorio**: se ordena por fecha y se separa el último 20 %
como test. Con 82 partidos jugados eso da ~65 de entrenamiento y ~17 de test —
un dataset muy pequeño para XGBoost, así que uso hiperparámetros conservadores
(pocos árboles, profundidad 2, bastante regularización L2) y comparo contra dos
referencias sencillas: predecir siempre la media de goles del entrenamiento, y
un modelo de Poisson lineal simple (solo `elo_diff` y fase). Si XGBoost no les
saca ventaja clara, es la señal de que se ha memorizado el train en vez de
aprender un patrón real.

In [18]:
from xgboost import XGBRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

FEATURE_COLS = ["home_elo", "away_elo", "elo_diff", "home_fifa_rank", "away_fifa_rank",
               "home_form_gf", "home_form_ga", "home_form_winrate",
               "away_form_gf", "away_form_ga", "away_form_winrate",
               "home_rest_days", "away_rest_days", "phase_ordinal"]
SIMPLE_COLS = ["elo_diff", "phase_ordinal"]

feat_sorted = features.sort_values("date").reset_index(drop=True)
split = int(len(feat_sorted) * 0.8)
train, test = feat_sorted.iloc[:split].copy(), feat_sorted.iloc[split:].copy()
print(f"Train: {len(train)} partidos ({train.date.min()} -> {train.date.max()}), "
     f"rondas: {train['round'].unique().tolist()}")
print(f"Test : {len(test)} partidos ({test.date.min()} -> {test.date.max()}), "
     f"rondas: {test['round'].unique().tolist()}")


def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


def train_xgb(target):
    model = XGBRegressor(n_estimators=100, max_depth=2, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         reg_lambda=5.0, min_child_weight=3, random_state=42)
    model.fit(train[FEATURE_COLS], train[target], sample_weight=train.competition_weight)
    return model


model_home = train_xgb("home_goals")
model_away = train_xgb("away_goals")

results = []
for name, target, model in [("XGBoost (local)", "home_goals", model_home),
                            ("XGBoost (visitante)", "away_goals", model_away)]:
    train_pred = model.predict(train[FEATURE_COLS])
    test_pred = model.predict(test[FEATURE_COLS])
    results.append({"modelo": name,
                    "MAE train": mean_absolute_error(train[target], train_pred),
                    "MAE test": mean_absolute_error(test[target], test_pred),
                    "RMSE test": rmse(test[target], test_pred)})

# referencias: media del train, y Poisson lineal simple (solo elo_diff + fase)
for name, target in [("Baseline media (local)", "home_goals"),
                     ("Baseline media (visitante)", "away_goals")]:
    mean_pred = train[target].mean()
    results.append({"modelo": name,
                    "MAE train": mean_absolute_error(train[target], [mean_pred] * len(train)),
                    "MAE test": mean_absolute_error(test[target], [mean_pred] * len(test)),
                    "RMSE test": rmse(test[target], [mean_pred] * len(test))})

for name, target in [("Poisson simple (local)", "home_goals"), ("Poisson simple (visitante)", "away_goals")]:
    pm = PoissonRegressor(alpha=1.0, max_iter=500)
    pm.fit(train[SIMPLE_COLS], train[target])
    train_pred, test_pred = pm.predict(train[SIMPLE_COLS]), pm.predict(test[SIMPLE_COLS])
    results.append({"modelo": name,
                    "MAE train": mean_absolute_error(train[target], train_pred),
                    "MAE test": mean_absolute_error(test[target], test_pred),
                    "RMSE test": rmse(test[target], test_pred)})

print()
print(pd.DataFrame(results).round(3).to_string(index=False))

Train: 65 partidos (2026-06-11 -> 2026-06-26), rondas: ['Group stage']
Test : 17 partidos (2026-06-26 -> 2026-07-01), rondas: ['Group stage', 'Round of 32']



                    modelo  MAE train  MAE test  RMSE test
           XGBoost (local)      0.657     0.889      1.056
       XGBoost (visitante)      0.384     0.734      0.965
    Baseline media (local)      1.356     0.892      1.060
Baseline media (visitante)      0.813     0.910      1.325
    Poisson simple (local)      1.121     0.800      0.937
Poisson simple (visitante)      0.730     0.774      0.926


In [19]:
imp = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importancia_local": model_home.feature_importances_,
    "importancia_visitante": model_away.feature_importances_,
}).sort_values("importancia_local", ascending=False)
print(imp.to_string(index=False))

          feature  importancia_local  importancia_visitante
         elo_diff           0.136874               0.145475
         home_elo           0.116879               0.073440
     away_form_ga           0.091635               0.101760
         away_elo           0.085460               0.062708
   home_fifa_rank           0.071568               0.134720
   away_rest_days           0.070960               0.000000
   home_rest_days           0.070737               0.032664
   away_fifa_rank           0.067895               0.103083
     home_form_ga           0.067198               0.080431
away_form_winrate           0.067037               0.085535
home_form_winrate           0.055890               0.052740
     home_form_gf           0.053914               0.072215
     away_form_gf           0.043952               0.055230
    phase_ordinal           0.000000               0.000000


## 12. `predict_match()` — probabilidades vía Poisson

Con las dos λ (goles esperados) que dan los modelos, se construye la matriz de
probabilidad conjunta de marcador asumiendo que los goles de cada equipo son
Poisson independientes: `P(local=i, visitante=j) = Poisson(i; λ_local) · Poisson(j; λ_visitante)`,
truncada a `max_goals` y renormalizada. De ahí salen el 1X2 (sumando por encima/
debajo/en la diagonal de la matriz) y los marcadores más probables.

In [20]:
import math

def _poisson_pmf(k, lam):
    return math.exp(-lam) * lam ** k / math.factorial(k)


def predict_match(home_team, away_team, phase, max_goals=7):
    """Predicción de un cruce: lambdas de XGBoost + matriz de probabilidad de Poisson."""
    h_str, a_str = team_strength(home_team), team_strength(away_team)
    h_form, a_form = team_pre_wc_form(home_team), team_pre_wc_form(away_team)
    last_played = pd.to_datetime(wc[wc.status == "played"].date).max()
    h_rest = team_rest_days(home_team, last_played + pd.Timedelta(days=1), wc)
    a_rest = team_rest_days(away_team, last_played + pd.Timedelta(days=1), wc)
    median_rest = pd.concat([features.home_rest_days, features.away_rest_days]).median()

    row = pd.DataFrame([{
        "home_elo": h_str["elo"], "away_elo": a_str["elo"],
        "elo_diff": h_str["elo"] - a_str["elo"],
        "home_fifa_rank": h_str["fifa_rank"], "away_fifa_rank": a_str["fifa_rank"],
        "home_form_gf": h_form["form_gf"], "home_form_ga": h_form["form_ga"],
        "home_form_winrate": h_form["form_winrate"],
        "away_form_gf": a_form["form_gf"], "away_form_ga": a_form["form_ga"],
        "away_form_winrate": a_form["form_winrate"],
        "home_rest_days": h_rest if h_rest is not None else median_rest,
        "away_rest_days": a_rest if a_rest is not None else median_rest,
        "phase_ordinal": ROUND_ORDINAL.get(phase, ROUND_ORDINAL["Group stage"]),
    }])[FEATURE_COLS]

    lam_home = max(float(model_home.predict(row)[0]), 0.05)
    lam_away = max(float(model_away.predict(row)[0]), 0.05)

    probs = pd.DataFrame(
        [[_poisson_pmf(i, lam_home) * _poisson_pmf(j, lam_away) for j in range(max_goals + 1)]
         for i in range(max_goals + 1)])
    probs /= probs.values.sum()  # renormaliza la cola truncada en max_goals

    p_home_win = sum(probs.iloc[i, j] for i in range(max_goals + 1) for j in range(max_goals + 1) if i > j)
    p_draw = sum(probs.iloc[i, i] for i in range(max_goals + 1))
    p_away_win = 1 - p_home_win - p_draw

    scorelines = sorted(
        ((f"{i}-{j}", probs.iloc[i, j]) for i in range(max_goals + 1) for j in range(max_goals + 1)),
        key=lambda x: -x[1])[:5]

    return {
        "home_team": home_team, "away_team": away_team, "phase": phase,
        "lambda_home": lam_home, "lambda_away": lam_away,
        "p_home_win": p_home_win, "p_draw": p_draw, "p_away_win": p_away_win,
        "top_scorelines": scorelines,
    }

In [21]:
pending = bracket_state[(bracket_state.confirmed_teams) & (bracket_state.status != "played")]
clean_pending = pending[pending.discrepancy.isna()]
demo = (clean_pending if len(clean_pending) else pending).iloc[0]
print(f"Cruce de ejemplo: Partido {demo.match_number} ({demo['round']}) "
     f"{demo.home_team} vs {demo.away_team} — status actual: {demo.status}")

result = predict_match(demo.home_team, demo.away_team, demo["round"])

print(f"\n{result['home_team']} vs {result['away_team']} ({result['phase']})")
print(f"Goles esperados (λ): {result['home_team']} {result['lambda_home']:.2f} "
     f"- {result['lambda_away']:.2f} {result['away_team']}")
print(f"\nP(1) {result['home_team']} gana : {result['p_home_win']:.1%}")
print(f"P(X) empate                     : {result['p_draw']:.1%}")
print(f"P(2) {result['away_team']} gana : {result['p_away_win']:.1%}")
print("\nMarcadores más probables:")
for score, p in result["top_scorelines"]:
    print(f"  {score}  ->  {p:.1%}")

Cruce de ejemplo: Partido 83 (Round of 32) Portugal vs Croatia — status actual: scheduled

Portugal vs Croatia (Round of 32)
Goles esperados (λ): Portugal 2.60 - 1.31 Croatia

P(1) Portugal gana : 65.0%
P(X) empate                     : 17.5%
P(2) Croatia gana : 17.6%

Marcadores más probables:
  2-1  ->  8.9%
  3-1  ->  7.7%
  1-1  ->  6.9%
  2-0  ->  6.8%
  3-0  ->  5.9%


---
## Anexo: alternativa descartada — scraper de FBref con nodriver (julio 2026)

Código del scraper original que superaba el desafío de Cloudflare de FBref con
un navegador Edge real controlado por `nodriver`. **Descartado** por frágil y
probablemente contrario a los términos de servicio de FBref; se conserva por si
hiciera falta en el futuro. Las celdas siguientes solo **definen** funciones
(no abren el navegador ni piden nada); la llamada final está comentada.

Requiere `pip install nodriver`. Detalle que resolvía: FBref esconde tablas en
comentarios HTML, pero al guardar el DOM renderizado por el navegador llegaban
ya «des-comentadas».

In [22]:
# --- capa de descarga con navegador (nodriver + Edge, event loop en un hilo) ---
import asyncio, threading
import nodriver as uc

FBREF_BASE = "https://fbref.com"
FBREF_SCHEDULE_URL = FBREF_BASE + "/en/comps/1/schedule/World-Cup-Scores-and-Fixtures"
FBREF_DELAY = 6  # FBref banea a partir de ~10 peticiones/minuto
_BROWSER_CANDIDATES = [
    r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
    r"C:\Program Files\Microsoft\Edge\Application\msedge.exe",
    r"C:\Program Files\Google\Chrome\Application\chrome.exe",
]
BROWSER_PATH = next((p for p in _BROWSER_CANDIDATES if Path(p).exists()), None)

_loop = None
_browser = None
_last_request = 0.0


def _get_loop():
    global _loop
    if _loop is None:
        _loop = asyncio.ProactorEventLoop()  # en Windows, necesario para subprocesos
        threading.Thread(target=_loop.run_forever, daemon=True).start()
    return _loop


def _run(coro, timeout=240):
    return asyncio.run_coroutine_threadsafe(coro, _get_loop()).result(timeout)


async def _get_browser():
    global _browser
    if _browser is None:
        _browser = await uc.start(browser_executable_path=BROWSER_PATH, headless=False)
    return _browser


def close_browser():
    global _browser
    if _browser is not None:
        res = _browser.stop()
        if asyncio.iscoroutine(res):
            _run(res)
        _browser = None


async def _fetch(url):
    global _last_request
    browser = await _get_browser()
    wait = FBREF_DELAY - (time.monotonic() - _last_request)
    if wait > 0:
        await asyncio.sleep(wait)
    page = await browser.get(url)
    html = ""
    for _ in range(30):  # hasta ~90 s por si el desafío de Cloudflare tarda
        await asyncio.sleep(3)
        html = await page.get_content()
        if "Just a moment" not in html[:3000] and len(html) > 30000:
            break
    _last_request = time.monotonic()
    if "Just a moment" in html[:3000]:
        raise RuntimeError(f"Cloudflare no resuelto para {url}")
    return html


def fetch_with_cache(url, force=False):
    p = cache_path(url)
    if p.exists() and not force:
        return p.read_text(encoding="utf-8")
    html = _run(_fetch(url))
    p.write_text(html, encoding="utf-8")
    return html

In [23]:
# --- parseo del calendario y los match reports de FBref ---
SCORE_RE = re.compile(r"(?:\((\d+)\))?\s*(\d+)\s*[–\-]\s*(\d+)\s*(?:\((\d+)\))?")
SUMMARY_STATS = ["shots", "shots_on_target", "fouls", "offsides", "crosses",
                 "cards_yellow", "cards_red", "tackles_won", "interceptions"]


def _clean_team(cell):
    a = cell.find("a")
    return (a or cell).get_text(strip=True)


def parse_schedule(html):
    """Tabla Scores & Fixtures de FBref -> DataFrame (una fila por partido)."""
    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table", id="sched_all") or soup.find(
        "table", id=lambda x: x and x.startswith("sched"))
    if table is None:
        raise ValueError("No se encontró la tabla de calendario")
    rows = []
    for tr in table.find("tbody").find_all("tr"):
        if tr.get("class") and "thead" in tr["class"]:
            continue
        cells = {c.get("data-stat"): c for c in tr.find_all(["th", "td"])}
        home_c, away_c = cells.get("home_team"), cells.get("away_team")
        if home_c is None or not home_c.get_text(strip=True):
            continue
        m = SCORE_RE.search(cells["score"].get_text(strip=True)) if cells.get("score") else None
        hp, hg, ag, ap = m.groups() if m else (None, None, None, None)
        match_url = None
        mr = cells.get("match_report")
        if mr and mr.get_text(strip=True) == "Match Report" and mr.find("a"):
            match_url = FBREF_BASE + mr.find("a")["href"]
        att = cells["attendance"].get_text(strip=True).replace(",", "") if cells.get("attendance") else ""
        time_txt = cells["start_time"].get_text(strip=True) if cells.get("start_time") else ""
        rows.append({
            "match_id": match_url.split("/matches/")[1].split("/")[0] if match_url else None,
            "date": cells["date"].get_text(strip=True) if cells.get("date") else "",
            "time_local": time_txt.split("(")[0],
            "round": cells["round"].get_text(strip=True) if cells.get("round") else "",
            "gameweek": cells["gameweek"].get_text(strip=True) if cells.get("gameweek") else "",
            "home_team": _clean_team(home_c),
            "away_team": _clean_team(away_c),
            "home_goals": int(hg) if hg is not None else None,
            "away_goals": int(ag) if ag is not None else None,
            "home_pens": int(hp) if hp is not None else None,
            "away_pens": int(ap) if ap is not None else None,
            "venue": cells["venue"].get_text(strip=True) if cells.get("venue") else "",
            "attendance": int(att) if att.isdigit() else None,
            "referee": cells["referee"].get_text(strip=True) if cells.get("referee") else "",
            "notes": cells["notes"].get_text(strip=True) if cells.get("notes") else "",
            "match_url": match_url,
            "status": "played" if m else "scheduled",
        })
    return pd.DataFrame(rows)


def _summary_totals(table):
    out = {}
    tf = table.find("tfoot")
    if not tf:
        return out
    for c in tf.find_all(["th", "td"]):
        ds = c.get("data-stat")
        if ds in SUMMARY_STATS:
            txt = c.get_text(strip=True)
            out[ds] = int(txt) if txt.lstrip("-").isdigit() else None
    return out


def _possession(soup):
    div = soup.find("div", id="team_stats")
    if not div:
        return None, None
    for tr in div.find_all("tr"):
        if tr.get_text(strip=True) == "Possession":
            vals = re.findall(r"(\d+)%", tr.find_next_sibling("tr").get_text())
            if len(vals) >= 2:
                return int(vals[0]), int(vals[1])
    return None, None


def _team_stats_extra(soup):
    out = {}
    div = soup.find("div", id="team_stats_extra")
    if not div:
        return out
    for block in div.find_all("div", recursive=False):
        vals = [d.get_text(strip=True) for d in block.find_all("div", recursive=False)]
        trip = vals[3:]  # los tres primeros divs son la cabecera
        for i in range(0, len(trip) - 2, 3):
            h, name, a = trip[i], trip[i + 1], trip[i + 2]
            if name and h.isdigit() and a.isdigit():
                out[name.lower()] = (int(h), int(a))
    return out


def _saves(soup, team_id):
    t = soup.find("table", id=f"keeper_stats_{team_id}")
    if not t or not t.find("tbody"):
        return None
    vals = [c.get_text(strip=True) for c in t.find("tbody").find_all("td", {"data-stat": "gk_saves"})]
    vals = [int(v) for v in vals if v.isdigit()]
    return sum(vals) if vals else None


def parse_match_report(html):
    """Match report de FBref -> dict {home_*/away_*} de stats de equipo."""
    soup = BeautifulSoup(html, "lxml")
    stats = {}
    tables = soup.find_all("table", id=re.compile(r"^stats_[0-9a-f]+_summary$"))
    team_ids = []
    for side, table in zip(("home", "away"), tables[:2]):
        team_ids.append(table["id"].split("_")[1])
        for kk, v in _summary_totals(table).items():
            stats[f"{side}_{kk}"] = v
    stats["home_possession"], stats["away_possession"] = _possession(soup)
    extra = _team_stats_extra(soup)
    for name in ("corners", "fouls", "offsides", "crosses", "interceptions"):
        if name in extra:
            h, a = extra[name]
            if stats.get(f"home_{name}") is None:
                stats[f"home_{name}"] = h
            if stats.get(f"away_{name}") is None:
                stats[f"away_{name}"] = a
    for side, tid in zip(("home", "away"), team_ids):
        stats[f"{side}_saves"] = _saves(soup, tid)
    return stats


def update_dataset_fbref(csv_path=CSV_PATH, refresh_schedule=True):
    """Versión original: construye el CSV scrapeando FBref con el navegador."""
    sched = parse_schedule(fetch_with_cache(FBREF_SCHEDULE_URL, force=refresh_schedule))
    old_stats = {}
    if Path(csv_path).exists():
        old = pd.read_csv(csv_path)
        for _, r in old.iterrows():
            if r.get("stats_scraped") == True and isinstance(r.get("match_id"), str):
                old_stats[r["match_id"]] = {c: r[c] for c in STAT_COLS if c in old.columns}
    pending = sched[(sched["status"] == "played") & (~sched["match_id"].isin(old_stats))]
    print(f"Jugados: {(sched['status'] == 'played').sum()} | nuevos a scrapear: {len(pending)}")
    all_stats = dict(old_stats)
    for i, (_, m) in enumerate(pending.iterrows(), 1):
        print(f"  [{i}/{len(pending)}] {m['date']}  {m['home_team']} vs {m['away_team']}", flush=True)
        try:
            s = parse_match_report(fetch_with_cache(m["match_url"]))
        except Exception as e:
            print(f"    ERROR: {e}")
            continue
        if s.get("home_possession") is not None or s.get("home_shots") is not None:
            all_stats[m["match_id"]] = s
    df = sched.copy()
    for c in STAT_COLS:
        df[c] = None
    df["stats_scraped"] = False
    for mid, s in all_stats.items():
        mask = df["match_id"] == mid
        if not mask.any():
            continue
        for c in STAT_COLS:
            v = s.get(c)
            df.loc[mask, c] = None if v is None or pd.isna(v) else v
        df.loc[mask, "stats_scraped"] = True
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    df.to_csv(csv_path, index=False)
    return df


# DESCARTADO — descomenta bajo tu responsabilidad (abre una ventana de Edge):
# df_fbref = update_dataset_fbref()
# close_browser()